# TF-c · Một vòng PINN cho dầm

Đi từ mạng → điểm miền/biên → đạo hàm → loss → một bước Adam → nhiều bước → kiểm chứng. Giải u''''=1, u(0)=u(1)=u''(0)=u''(1)=0. Nghiệm đúng chỉ xuất hiện ở bước kiểm chứng.


In [ ]:
import tensorflow as tf
tf.keras.backend.set_floatx('float64')  # đặt trước khi tạo Dense để đồng nhất dtype
tf.keras.utils.set_random_seed(7)
model = tf.keras.Sequential([tf.keras.Input(shape=(1,)),
    tf.keras.layers.Dense(16, activation='tanh'),
    tf.keras.layers.Dense(16, activation='tanh'),
    tf.keras.layers.Dense(1)])
x_c = tf.reshape((tf.cast(tf.range(16),tf.float64)+0.5)/16,(-1,1))
x_b = tf.constant([[0.0],[1.0]],dtype=tf.float64)
print(x_c.shape,x_b.shape)


## 1. Đạo hàm theo x

Bốn tape lấy u''''. Chúng phải nằm trong tape_theta của bước train để loss còn truyền gradient về trọng số.


In [ ]:
def derivatives(model,x):
    # Tape lồng nhau lấy u', u'', u''' rồi u'''' theo tọa độ.
    with tf.GradientTape() as t4:
        t4.watch(x)  # x hằng không được theo dõi mặc định
        with tf.GradientTape() as t3:
            t3.watch(x)
            with tf.GradientTape() as t2:
                t2.watch(x)
                with tf.GradientTape() as t1:
                    t1.watch(x)
                    u=model(x)
                u1=t1.gradient(u,x)
            u2=t2.gradient(u1,x)
        u3=t3.gradient(u2,x)
    u4=t4.gradient(u3,x)
    return u,u2,u4


## 2. Một bước train

Loss PDE là trung bình bình phương u''''-1. Loss biên cộng bình phương u và u'' ở hai đầu. Tape ngoài lấy đạo hàm loss theo trọng số; Adam cập nhật chúng.


In [ ]:
optimizer=tf.keras.optimizers.Adam(learning_rate=0.005)
def train_step():
    with tf.GradientTape() as tape_theta:  # ghi loss theo trọng số
        _,_,u4_c=derivatives(model,x_c)
        u_b,u2_b,_=derivatives(model,x_b)
        loss_pde=tf.reduce_mean((u4_c-1.0)**2)  # MSE residual PDE
        loss_bc=tf.reduce_sum(u_b**2+u2_b**2)  # điều kiện biên
        loss=loss_pde+10.0*loss_bc
    weights=model.trainable_variables
    gradients=tape_theta.gradient(loss,weights)  # d(loss)/d(weights)
    optimizer.apply_gradients(zip(gradients,weights))  # cập nhật một step
    return float(loss_pde),float(loss_bc),float(loss)
print('một bước:',train_step())


## 3. Lặp và kiểm chứng

Mỗi bước forward lại với trọng số vừa cập nhật. Lưới kiểm chứng mới không được dùng để train.


In [ ]:
history=[]
for step in range(150):
    _,_,loss=train_step()  # một lần cập nhật trên toàn bộ điểm miền
    history.append(loss)
print('loss đầu/cuối:',history[0],history[-1])
x_test=tf.reshape(tf.linspace(tf.constant(0.,tf.float64),tf.constant(1.,tf.float64),101),(-1,1))
u_pred=model(x_test)
u_exact=(x_test**4-2*x_test**3+x_test)/24  # chỉ dùng kiểm chứng
error=tf.sqrt(tf.reduce_sum((u_pred-u_exact)**2)/tf.reduce_sum(u_exact**2))
print('sai số chuyển vị tương đối:',float(error))


Tự làm: chỉ đổi 16 điểm miền thành 32 hoặc 150 bước thành 300. Ghi loss cuối và sai số kiểm chứng. Loss PDE không phải sai số nghiệm. Đọc trang TensorFlow đầy đủ và script để thêm mômen, CSV, nhiều seed.
